# 자연어 질문에서 matadata filtering 생성하기
- 자연어 질문 검색 조건으로 변환하고 변환된 결과를 pinecone metadata filtering으로 직접 조합한다.

## 환결 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

# PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_META_INDEX_NAME = 'adv-meta-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [2]:
import pandas as pd

document_df = pd.read_csv("data/documents_meta.csv")
queries_df = pd.read_csv("data/queries_meta.csv")

document_df.head()

,doc_id,title,content,author,category
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...",박민준,여행;제주;관광
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기...",이서연,음식;비빔밥;역사
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet...",최유나,연예;음악;대중문화
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...,정하늘,역사;문화;문자
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...,정하늘,역사;군사;전쟁


In [3]:
queries_df.head()

,query_id,query_text,relevant_doc_ids
0,MQ1,김철수가 작성한 검색 카테고리 문서 중 ChromaDB와 Qdrant를 비교한 문서...,D27=3
1,MQ2,김철수 저자의 검색 카테고리 문서에서 Contextual Compression 개념...,D28=3
2,MQ3,김철수가 쓴 검색 카테고리 문서 중 Self-Query Retriever 원리를 다...,D29=3
3,MQ4,김철수 저자의 검색 카테고리 문서에서 Multi-Hop Retrieval 예시를 찾아줘,D30=3
4,MQ5,한지민 저자의 AI 카테고리 문서 중 기후 예측 연구 사례와 모델을 다룬 자료는?,D14=3;D26=2


## 벡터 스토어 준비

In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터 스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_META_INDEX_NAME,
    embedding=embeddings
)

## 출력 함수

In [18]:
def print_docs(docs):
    for doc in docs:
        print(f"{doc.metadata['doc_id']} : ")
        print('author : ',doc.metadata.get('author'))
        print('category : ',doc.metadata.get('category'))
        print(doc.page_content)
        print()

def doc_ids(docs):
    return [doc.metadata['doc_id'] for doc in docs]

## 자연어 질문을 구조화된 검색 조건으로 변환하기

In [5]:
from pydantic import BaseModel, Field
from typing import List,Optional
from langchain_openai import ChatOpenAI

class MetadataSearchQuery(BaseModel):

    query: str = Field(
        description=(
        '벡터 검색에 사용할 핵심 검색어'
        '작성자나 카테고리 조건은 제외하고, 문서 내용과 관련된 검색어만 작성한다.'
        )
    )

    author: Optional[str] = Field(
        default=None,
        description=(
            '문서를 작성한 저자 이름'
            '질문에 저자 조건이 없으면 Null로 둔다'
            '예: 김철수, 한지민, 이서연, 박민준, 최유나'
        )
    )

    categories: List[str] = Field(
        default_factory=list,
        description=(
            '문서가 속한 카테고리 조건 목록'
            '질문에 카테고리 조건이 없으면 빈 리스트로 둔다.'
            '예: 검색, RAG, 벡터DB, AI, 음식, 여행'
        )
    )

llm = ChatOpenAI(model=OPENAI_LLM_MODEL,temperature=0)
structured_llm = llm.with_structured_output(MetadataSearchQuery)

## 검색 조건 생성 함수

In [6]:
def genarate_search_query(user_query:str) -> MetadataSearchQuery:
    prompt = f'''
다음 상용자 질문을 벡터 검색 조건으로 변환하세요.

규칙:
1. query에는 문서 본문과 의미적으로 비교할 검색어만 작성하세요.
2. 작성자 조건은 author에 작성하세요.
3. 카테고리 조건은 category에 작성하세요.
4. 질문에 작성자 조건이 없으면 author는 null로 두세요.
5. 질문에 카테고리 조건이 없으면 category는 빈 리스트로 두세요.
6. '검색 카테고리', '카테고리가 검색'처럼 표현되면 categories에 '검색'을 넣으세요.
7. 'AI 관련 문서'처럼 표현되면 categories에 'AI'를 넣으세요.

사용자 질문:
{user_query}
'''
    return structured_llm.invoke(prompt)

In [7]:
user_query = '김철수가 작성한 검색 카테고리 문서 중 Chroma와 Qdrant를 비교한 문서를 찾아줘'

search_query = genarate_search_query(user_query)
search_query

MetadataSearchQuery(query='Chroma와 Qdrant 비교', author='김철수', categories=['검색'])

## 구조환 결과를 Pinecone filter로 변환

In [8]:
def build_pinecone_filter(search_query:MetadataSearchQuery):
    conditions = []

    if search_query.author:
        conditions.append({'author' : {'$eq' : search_query.author}})

    if search_query.categories:
        conditions.append({'category':{'$in':search_query.categories}})

    if len(conditions) == 0:
        return None
    
    if len(conditions) == 1:
        return conditions[0]
    
    return {'$and':conditions}

In [9]:
filter_dict = build_pinecone_filter(search_query)
filter_dict

{'$and': [{'author': {'$eq': '김철수'}}, {'category': {'$in': ['검색']}}]}

## 자연어 기반 metadata filter 검색 함수

In [19]:
def natural_language_filter_search(user_query:str,top_k:int=5,verbose:bool=True):
    search_query = genarate_search_query(user_query)
    filter_dict = build_pinecone_filter(search_query)

    docs = vector_store.similarity_search(
        search_query.query,
        k=top_k,
        filter=filter_dict
    )

    if verbose:
        print('원본 질문')
        print(user_query)
        print()
        print('생성된 검색어')
        print(search_query)
        print()
        print('추출된 author')
        print(search_query.author)
        print()
        print('추출된 category')
        print(search_query.categories)
        print()
        print('pinecone filter')
        print(filter_dict)
        print()
        print('검색 결과 id')
        print(doc_ids(docs))
        print()
        print('='*100)

    return search_query, filter_dict, docs

## 단일 질문으로 흐름 확인

In [20]:
user_query = '김철수가 작성한 검색 카테고리 문서 중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘'

search_query, filter_dict, docs =natural_language_filter_search(user_query)
print_docs(docs)

원본 질문
김철수가 작성한 검색 카테고리 문서 중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘

생성된 검색어
query='ChromaDB Qdrant 비교' author='김철수' categories=['검색']

추출된 author
김철수

추출된 category
['검색']

pinecone filter
{'$and': [{'author': {'$eq': '김철수'}}, {'category': {'$in': ['검색']}}]}

검색 결과 id
['D27', 'D29', 'D28', 'D30']

D27 : 
author :  김철수
category :  ['검색', '벡터DB', '기술']
ChromaDB와 Qdrant는 벡터 검색 라이브러리로, ChromaDB는 오픈소스 벡터 DB로 간단한 파이썬 인터페이스를 제공하며, Qdrant는 Rust 기반 고성능 벡터 DB로 GPU 가속 지원 및 필터링 기능이 강점입니다. 성능 비교 실험 시 인덱싱 속도, 검색 응답 속도, 메모리 사용량, 스케일링 용이성 등을 비교합니다. 파이썬 코드 예제와 벤치마크 결과가 공개되어 있어, 개발자가 선택하기 용이합니다.

D29 : 
author :  김철수
category :  ['검색', 'RAG', '메타데이터']
Self-Query Retriever는 문서 메타데이터(제목·요약·키워드)를 분석해, 사용자가 실제로 검색할 만한 쿼리를 GPT-4o-mini 등 생성형 모델로 생성한 뒤, 생성된 가상 쿼리를 다시 검색에 활용하는 기법입니다. 이 과정을 통해 사용자가 입력한 실제 질의보다 검색 품질을 높이는 효과를 얻을 수 있으며, 생성된 쿼리는 ‘Self-Query’라고 불립니다.

D28 : 
author :  김철수
category :  ['검색', 'RAG', '압축']
Contextual Compression은 긴 텍스트에서 핵심 정보만 추출해 압축(요약)한 뒤 검색 효율을 높이는 기법입니다. 예를 들어, 긴 문서를 PEGASUS 기반 한국어 요약 모델로 요

## 난이도별 질문

In [22]:
test_queries = [
    '김철수가 작성한 문서를 모두 찾아줘.',
    '카테고리가 검색인 무서들을 찾아줘.',
    '백준호 저자의 디자인패턴 카테고리 문서 중 싱글톤 패턴과 멀티스레드 안정성을 설명한 자료는?'
]

for query in test_queries:
    search_query, filter_dict, docs =natural_language_filter_search(query)

원본 질문
김철수가 작성한 문서를 모두 찾아줘.

생성된 검색어
query='' author='김철수' categories=[]

추출된 author
김철수

추출된 category
[]

pinecone filter
{'author': {'$eq': '김철수'}}

검색 결과 id
['D27', 'D28', 'D29', 'D30']

원본 질문
카테고리가 검색인 무서들을 찾아줘.

생성된 검색어
query='무서들' author=None categories=['검색']

추출된 author
None

추출된 category
['검색']

pinecone filter
{'category': {'$in': ['검색']}}

검색 결과 id
['D29', 'D28', 'D30', 'D27']

원본 질문
백준호 저자의 디자인패턴 카테고리 문서 중 싱글톤 패턴과 멀티스레드 안정성을 설명한 자료는?

생성된 검색어
query='싱글톤 패턴 멀티스레드 안정성 설명' author='백준호' categories=['디자인패턴']

추출된 author
백준호

추출된 category
['디자인패턴']

pinecone filter
{'$and': [{'author': {'$eq': '백준호'}}, {'category': {'$in': ['디자인패턴']}}]}

검색 결과 id
[]



## 전체 질의 실행
- queries_meta.csv으 전체 질문을 실행한다.

In [23]:
from tqdm import tqdm

nl_filter_results = {}
generate_conditions = {}

for idx,row in tqdm(queries_df.iterrows(), total=len(queries_df)):
    qid = row['query_id']
    query_text = row['query_text']

    search_query, filter_dict, docs = natural_language_filter_search(
        query_text,
        verbose=False
    )

    nl_filter_results[qid] = doc_ids(docs)
    generate_conditions[qid] = {
        'query' : search_query.query,
        'author' : search_query.author,
        'category' : search_query.categories,
        'filter' : filter_dict
    }

100%|██████████| 30/30 [00:53<00:00,  1.79s/it]


## 생성된 검색 조건 일부 확인

In [25]:
for qid in ['MQ1','MQ6','MQ12','MQ18','MQ25']:
    query_text = queries_df.loc[queries_df['query_id'] == qid, 'query_text'].iloc[0]

    print(qid,query_text)
    print(generate_conditions[qid])
    print('검색 결과 : ', nl_filter_results[qid])
    print('='*100)

MQ1 김철수가 작성한 검색 카테고리 문서 중 ChromaDB와 Qdrant를 비교한 문서를 찾아줘
{'query': 'ChromaDB Qdrant 비교', 'author': '김철수', 'category': ['검색'], 'filter': {'$and': [{'author': {'$eq': '김철수'}}, {'category': {'$in': ['검색']}}]}}
검색 결과 :  ['D27', 'D29', 'D28', 'D30']
MQ6 한지민이 작성한 AI 정책 카테고리 문서에서 인재 양성과 윤리 가이드라인 내용을 찾아줘
{'query': '인재 양성 윤리 가이드라인', 'author': '한지민', 'category': ['AI', '정책'], 'filter': {'$and': [{'author': {'$eq': '한지민'}}, {'category': {'$in': ['AI', '정책']}}]}}
검색 결과 :  ['D25', 'D14', 'D26', 'D6']
MQ12 박민준이 작성한 여행 카테고리 문서에서 서울 근교 당일치기 여행지를 추천한 자료는?
{'query': '서울 근교 당일치기 여행지 추천', 'author': '박민준', 'category': ['여행'], 'filter': {'$and': [{'author': {'$eq': '박민준'}}, {'category': {'$in': ['여행']}}]}}
검색 결과 :  ['D12', 'D1']
MQ18 박준호가 작성한 디자인패턴 카테고리 문서에서 Strategy 패턴의 결제 시스템 예제를 찾아줘
{'query': 'Strategy 패턴 결제 시스템 예제', 'author': '박준호', 'category': ['디자인패턴'], 'filter': {'$and': [{'author': {'$eq': '박준호'}}, {'category': {'$in': ['디자인패턴']}}]}}
검색 결과 :  ['D23', 'D24', 'D22']
MQ25 한지민 저자의 환경 카테고리 문서 중 2024년 기후 변

In [26]:
import numpy as np

def parse_relevant(relevant_str):
    """다중 정답 및 등급을 처리하기 위한 헬퍼 함수"""
    pairs = relevant_str.split(";")
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split("=")
        rel_dict[doc_id] = grade
    return rel_dict 

def compute_metrics(predicted, relevant_dict, k=5):
    relevant_docs = set(relevant_dict.keys())
    top_k = predicted[:k]
    hits = sum(1 for doc in top_k if doc in relevant_docs)
    precision = hits / k
    total_relevant = len(relevant_docs)
    recall = hits / total_relevant if total_relevant > 0 else 0 
    rr = 0
    for idx, doc in enumerate(top_k):
        if doc in relevant_docs:
            rr = 1 / (idx + 1)
            break
    num_correct = 0
    precision_sum = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_docs:
            num_correct += 1
            precision_sum += num_correct / (i + 1)
    denominator = min(total_relevant, k)
    ap = precision_sum / denominator if denominator > 0 else 0
    return precision, recall, rr, ap

def evaluate_all(method_results, queries_df, k=5):
    prec_list, rec_list, rr_list, ap_list = [], [], [], []
    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[qid]
        p, r, rr, ap = compute_metrics(predicted, relevant_dict, k)
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)
    return {
        'Precision@k' : np.mean(prec_list),
        'Recall@k' : np.mean(rec_list),
        'MRR' : np.mean(rr_list),
        'MAP' : np.mean(ap_list),
    }

## dense 검색과 자연어 기반 metadata filtering 검색 비교

In [27]:
dense_results = {}

for idx, row in tqdm(queries_df.iterrows(),total=len(queries_df)):
    qid = row['query_id']
    query_text = row['query_text']

    dense_docs = vector_store.similarity_search(query_text,k=5)
    dense_results[qid] = doc_ids(dense_docs)

100%|██████████| 30/30 [00:17<00:00,  1.67it/s]


In [28]:
dense_metrics = evaluate_all(dense_results,queries_df)
nl_filter_metrics = evaluate_all(nl_filter_results,queries_df)

metrics_df = pd.DataFrame({
    'Metric' : ["Precision@k","Recall@k","MRR","MAP"],
    'Dense' : [dense_metrics["Precision@k"],dense_metrics["Recall@k"],dense_metrics["MRR"],dense_metrics["MAP"]],
    'nl_filter' : [nl_filter_metrics["Precision@k"],nl_filter_metrics["Recall@k"],nl_filter_metrics["MRR"],nl_filter_metrics["MAP"]],
})
metrics_df

,Metric,Dense,nl_filter
0,Precision@k,0.253333,0.293333
1,Recall@k,0.941667,1.000000
2,MRR,0.941667,1.000000
3,MAP,0.887778,0.994444
